In [1]:
# ===============================================================
# SECTION 0 — IMPORTS & GLOBAL SETTINGS
# ===============================================================

import numpy as np
import pandas as pd
import warnings
warnings.filterwarnings("ignore")

from sklearn.model_selection import GroupKFold
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import (
    roc_auc_score,
    precision_recall_curve,
    f1_score,
    accuracy_score
)

from catboost import CatBoostClassifier
import lightgbm as lgb
import xgboost as xgb

import gc
import os
import json
import pickle

# Global random seed
SEED = 42
np.random.seed(SEED)

print("Imports loaded. Environment ready.")


Imports loaded. Environment ready.


In [2]:
# ===============================================================
# SECTION 1 — LOAD RAW DATA
# ===============================================================

df = pd.read_csv("cct_train.csv")

print("Data loaded:", df.shape)
print("Columns:", list(df.columns))


Data loaded: (700807, 26)
Columns: ['ssn', 'cc_num', 'first', 'last', 'gender', 'street', 'city', 'state', 'zip', 'lat', 'long', 'city_pop', 'job', 'dob', 'acct_num', 'profile', 'trans_num', 'trans_date', 'trans_time', 'unix_time', 'category', 'amt', 'merchant', 'merch_lat', 'merch_long', 'is_fraud']


In [3]:
# ===============================================================
# SECTION 2 — RAW CLEANUP, DATETIME, PROFILE SPLIT, BASE FEATURES
# ===============================================================

# ---------------------------
# 2.1 — Convert date/time cols
# ---------------------------
df["trans_datetime"] = pd.to_datetime(
    df["trans_date"].astype(str) + " " + df["trans_time"].astype(str)
)

df = df.sort_values(["cc_num", "trans_datetime"]).reset_index(drop=True)

df["prev_ts"] = df.groupby("cc_num")["trans_datetime"].shift()
df["time_diff"] = (df["trans_datetime"] - df["prev_ts"]).dt.total_seconds()
df["time_diff"] = df["time_diff"].fillna(df["time_diff"].median())

df["hour"] = df["trans_datetime"].dt.hour
df["day"] = df["trans_datetime"].dt.day
df["weekday"] = df["trans_datetime"].dt.weekday

# ---------------------------
# 2.2 — Demographic profile split
# ---------------------------
demo = df["profile"].str.split("_", expand=True)
demo.columns = ["life_stage", "age_band", "profile_gender", "area_type"]
df = pd.concat([df, demo], axis=1)

# ---------------------------
# 2.3 — Merchant features
# ---------------------------
df["merchant_freq"] = df.groupby("merchant")["merchant"].transform("count")

df["merchant_avg_amt"] = df.groupby("merchant")["amt"].transform("mean")
df["merchant_std_amt"] = df.groupby("merchant")["amt"].transform("std").fillna(0)

df["merchant_user_freq"] = df.groupby(["cc_num", "merchant"])["amt"].transform("count")

# ---------------------------
# 2.4 — User global stats
# ---------------------------
grouped_user = df.groupby("cc_num")

df["user_avg_amt"] = grouped_user["amt"].transform("mean")
df["user_std_amt"] = grouped_user["amt"].transform("std").fillna(0)
df["user_max_amt"] = grouped_user["amt"].transform("max")
df["user_min_amt"] = grouped_user["amt"].transform("min")
df["user_trans_ct"] = grouped_user["amt"].transform("count")

print("SECTION 2 complete — all base features, datetime, demographics, merchant stats added.")


SECTION 2 complete — all base features, datetime, demographics, merchant stats added.


In [4]:
# ===============================================================
# SECTION 3 — HAVERSINE DISTANCE (GEO-FEATURE)
# ===============================================================

def haversine(lat1, lon1, lat2, lon2):
    """
    Computes the great-circle distance between two (lat, lon) pairs.
    Result is in kilometers.
    """
    R = 6371  # Earth radius (km)
    lat1, lon1, lat2, lon2 = map(np.radians, [lat1, lon1, lat2, lon2])
    dlat = lat2 - lat1
    dlon = lon2 - lon1
    a = (
        np.sin(dlat/2)**2 
        + np.cos(lat1) * np.cos(lat2) * np.sin(dlon/2)**2
    )
    return 2 * R * np.arcsin(np.sqrt(a))

df["distance_km"] = haversine(
    df["lat"], df["long"],
    df["merch_lat"], df["merch_long"]
)


In [5]:
# ===============================================================
# SECTION 4 — SPARKOV BEHAVIOURAL FEATURES (USER PATTERNS)
# ===============================================================

grouped_user = df.groupby("cc_num")

# ---------------------------------------------------------------
# 4.1 — USER HOME LOCATION (modal lat/long)
# ---------------------------------------------------------------
user_home_lat = grouped_user["lat"].agg(lambda x: x.mode()[0] if len(x.mode()) else x.iloc[0])
user_home_long = grouped_user["long"].agg(lambda x: x.mode()[0] if len(x.mode()) else x.iloc[0])

df = df.merge(user_home_lat.rename("user_home_lat"), left_on="cc_num", right_index=True)
df = df.merge(user_home_long.rename("user_home_long"), left_on="cc_num", right_index=True)

df["dist_from_home"] = np.sqrt(
    (df["lat"] - df["user_home_lat"])**2 +
    (df["long"] - df["user_home_long"])**2
)

# ---------------------------------------------------------------
# 4.2 — USER–CATEGORY FREQUENCY + CATEGORY RARITY
# ---------------------------------------------------------------
df["user_category_freq"] = df.groupby(["cc_num", "category"])["category"].transform("count")
df["category_rarity"] = 1 / (df["user_category_freq"] + 1)

# ---------------------------------------------------------------
# 4.3 — FIRST-TIME MERCHANT INDICATOR
# ---------------------------------------------------------------
df["is_first_time_merchant"] = (
    df.groupby(["cc_num", "merchant"]).cumcount().eq(0).astype(int)
)

# ---------------------------------------------------------------
# 4.4 — USER MEAN TRANSACTION HOUR & HOUR DEVIATION
# ---------------------------------------------------------------
user_mean_hour = grouped_user["hour"].mean().rename("user_mean_hour")
df = df.merge(user_mean_hour, left_on="cc_num", right_index=True)

df["hour_deviation"] = (df["hour"] - df["user_mean_hour"]).abs()

# ---------------------------------------------------------------
# 4.5 — BASIC SPEND VELOCITY FEATURES
# ---------------------------------------------------------------
df["amt_to_user_avg_ratio"] = df["amt"] / (df["user_avg_amt"] + 1e-6)
df["amt_to_user_std_ratio"] = df["amt"] / (df["user_std_amt"] + 1e-6)
df["is_high_velocity"] = (df["amt_to_user_avg_ratio"] > 2.5).astype(int)


In [6]:
# ===============================================================
# SECTION 5 — ADVANCED DYNAMIC FEATURES
# ===============================================================

grouped_user = df.groupby("cc_num")

# ---------------------------------------------------------------
# 5.1 — Rolling amount features (window = 3)
# ---------------------------------------------------------------
df["rolling_amt_mean_3"] = (
    grouped_user["amt"].rolling(3, min_periods=1).mean()
    .reset_index(level=0, drop=True)
)

df["rolling_amt_std_3"] = (
    grouped_user["amt"].rolling(3, min_periods=1).std()
    .fillna(0).reset_index(level=0, drop=True)
)

df["rolling_amt_max_3"] = (
    grouped_user["amt"].rolling(3, min_periods=1).max()
    .reset_index(level=0, drop=True)
)

# ---------------------------------------------------------------
# 5.2 — Rolling time-diff features (window = 3)
# ---------------------------------------------------------------
df["rolling_time_diff_mean_3"] = (
    grouped_user["time_diff"].rolling(3, min_periods=1).mean()
    .reset_index(level=0, drop=True)
)

df["rolling_time_diff_std_3"] = (
    grouped_user["time_diff"].rolling(3, min_periods=1).std()
    .fillna(0).reset_index(level=0, drop=True)
)

# ---------------------------------------------------------------
# 5.3 — User–Merchant interaction frequency
# ---------------------------------------------------------------
df["user_merchant_ct"] = df.groupby(["cc_num", "merchant"]).cumcount()

# ---------------------------------------------------------------
# 5.4 — User–Merchant average amount (expanding mean)
# ---------------------------------------------------------------
df["user_merchant_avg_amt"] = (
    df.groupby(["cc_num", "merchant"])["amt"]
    .expanding().mean().reset_index(level=[0,1], drop=True)
)

# ---------------------------------------------------------------
# 5.5 — User–Merchant time since last transaction
# ---------------------------------------------------------------
df["user_merchant_last_ts"] = (
    df.groupby(["cc_num", "merchant"])["trans_datetime"].shift()
)

df["user_merchant_time_since_last"] = (
    (df["trans_datetime"] - df["user_merchant_last_ts"]).dt.total_seconds()
)

df["user_merchant_time_since_last"] = (
    df["user_merchant_time_since_last"]
    .fillna(df["user_merchant_time_since_last"].median())
)

# ---------------------------------------------------------------
# 5.6 — Deviation from user's normal behaviour
# ---------------------------------------------------------------
df["dev_from_user_avg"] = df["amt"] - df["user_avg_amt"]
df["amt_user_avg_ratio"] = df["amt"] / (df["user_avg_amt"] + 1e-6)

df["is_spike_3x"] = (df["amt"] > 3 * df["user_avg_amt"]).astype(int)
df["is_spike_5x"] = (df["amt"] > 5 * df["user_avg_amt"]).astype(int)

# ---------------------------------------------------------------
# 5.7 — Cumulative amount features
# ---------------------------------------------------------------
df["cumulative_amt"] = df.groupby("cc_num")["amt"].cumsum()
df["cumulative_avg_amt"] = df["cumulative_amt"] / (df["user_trans_ct"] + 1e-6)
df["amt_cumavg_ratio"] = df["amt"] / (df["cumulative_avg_amt"] + 1e-6)

# ---------------------------------------------------------------
# 5.8 — User-category rarity (additional)
# ---------------------------------------------------------------
df["user_category_count"] = (
    df.groupby(["cc_num", "category"])["category"].transform("count")
)

df["category_rarity_for_user"] = 1 / (df["user_category_count"] + 1)

# ---------------------------------------------------------------
# 5.9 — Night-time behaviour & transitions
# ---------------------------------------------------------------
df["is_night"] = ((df["hour"] <= 5) | (df["hour"] >= 23)).astype(int)

df["night_to_day_jump"] = (
    (df["is_night"].shift(1) == 1) &
    (df["is_night"] == 0)
).astype(int).fillna(0)


In [7]:
# ===============================================================
# SECTION 6 — CLEANUP & FINAL FEATURE MATRIX (X, y)
# ===============================================================

# ---------------------------------------------------------------
# 6.1 — Columns to drop (PII + leakage + raw timestamp fields)
# ---------------------------------------------------------------
drop_cols = [
    "ssn", "first", "last", "street", "city", "state", "zip",
    "job", "dob", "acct_num", "trans_num",
    "trans_date", "trans_time", "unix_time",
    "merchant", "profile", "user_merchant_last_ts", "prev_ts"
]

df2 = df.drop(columns=drop_cols)

# ---------------------------------------------------------------
# 6.2 — Define target and features
# ---------------------------------------------------------------
y = df2["is_fraud"]
X = df2.drop(columns=["is_fraud", "trans_datetime"])  # remove raw datetime

# ---------------------------------------------------------------
# 6.3 — Categorical feature columns
# ---------------------------------------------------------------
categorical_cols = [
    "gender",
    "category",
    "life_stage",
    "age_band",
    "profile_gender",
    "area_type",
]

# For CatBoost, these names are used directly
cat_features = [X.columns.get_loc(col) for col in categorical_cols]

# ---------------------------------------------------------------
# 6.4 — Group labels for GroupKFold (customer-level leakage prevention)
# ---------------------------------------------------------------
groups = df2["cc_num"]


In [8]:
# ===============================================================
# SECTION 7 — LABEL ENCODING FOR LGBM & XGBOOST
# ===============================================================

# Copy X so CatBoost can still use raw categorical strings
X_lgb = X.copy()
X_xgb = X.copy()

label_encoders = {}

for col in categorical_cols:
    le = LabelEncoder()
    X_lgb[col] = le.fit_transform(X_lgb[col].astype(str))
    X_xgb[col] = le.transform(X_xgb[col].astype(str))
    label_encoders[col] = le


In [9]:
# ===============================================================
# SECTION 8 — LEAK-SAFE MERCHANT ENCODINGS
# ===============================================================

def add_fold_safe_merchant_encodings(X_train, X_val, y_train):
    """
    Creates merchant-fraud-rate encodings using only the TRAIN fold.
    Validation fold never leaks its labels into the encoding.
    """

    # Build temporary training table
    temp = pd.DataFrame({
        "merchant_freq": X_train["merchant_freq"],
        "y": y_train
    })

    # Quantile bin merchant frequency
    temp["freq_bin"] = pd.qcut(
        temp["merchant_freq"], 
        q=20, 
        duplicates="drop"
    )

    # Fraud rate per frequency bin
    fraud_rate = temp.groupby("freq_bin")["y"].mean().to_dict()

    # TRAIN → map bins to fraud rate
    X_train["merchant_fraud_rate"] = (
        temp["freq_bin"].map(fraud_rate).astype(float)
    )

    # VAL → compute bins using TRAIN quantiles
    X_val_bins = pd.qcut(
        X_val["merchant_freq"],
        q=20,
        duplicates="drop"
    )

    # VAL → map bins to fraud rate
    X_val["merchant_fraud_rate"] = (
        X_val_bins.map(fraud_rate).astype(float)
    )

    # Fallback for unseen bins
    X_val["merchant_fraud_rate"] = (
        X_val["merchant_fraud_rate"]
        .fillna(y_train.mean())
        .astype(float)
    )

    return X_train, X_val


In [10]:
# ===============================================================
# SECTION 9 — TRAIN ALL MODELS WITH GROUP K-FOLD
# ===============================================================

N_FOLDS = 5
gkf = GroupKFold(n_splits=N_FOLDS)

# Storage for fold models
cat_models = []
lgb_models = []
xgb_models = []

# Storage for metrics
cat_metrics = []
lgb_metrics = []
xgb_metrics = []


# ===============================================================
# Helper: Compute best F1 threshold from probabilities
# ===============================================================
def compute_best_threshold(y_true, y_prob):
    precision, recall, thresholds = precision_recall_curve(y_true, y_prob)
    f1_scores = 2 * precision * recall / (precision + recall + 1e-9)
    idx = np.argmax(f1_scores)
    return thresholds[idx], f1_scores[idx]


# ===============================================================
# Begin cross-validation
# ===============================================================
for fold, (tr_idx, val_idx) in enumerate(gkf.split(X, y, groups)):
    
    # --------- Split main matrices ---------
    Xtr_cat = X.iloc[tr_idx].copy()
    Xva_cat = X.iloc[val_idx].copy()
    
    Xtr_lgb = X_lgb.iloc[tr_idx].copy()
    Xva_lgb = X_lgb.iloc[val_idx].copy()
    
    Xtr_xgb = X_xgb.iloc[tr_idx].copy()
    Xva_xgb = X_xgb.iloc[val_idx].copy()
    
    ytr = y.iloc[tr_idx].copy()
    yva = y.iloc[val_idx].copy()

    # --------- Apply leak-safe merchant encoding ---------
    Xtr_cat, Xva_cat = add_fold_safe_merchant_encodings(Xtr_cat, Xva_cat, ytr)
    Xtr_lgb, Xva_lgb = add_fold_safe_merchant_encodings(Xtr_lgb, Xva_lgb, ytr)
    Xtr_xgb, Xva_xgb = add_fold_safe_merchant_encodings(Xtr_xgb, Xva_xgb, ytr)

    # ===============================================================
    # 9A — CATBOOST
    # ===============================================================
    cat_model = CatBoostClassifier(
        iterations=700,
        depth=8,
        learning_rate=0.05,
        loss_function="Logloss",
        eval_metric="AUC",
        random_seed=SEED,
        verbose=False
    )
    cat_model.fit(Xtr_cat, ytr, eval_set=(Xva_cat, yva), cat_features=cat_features, verbose=False)

    p_cat = cat_model.predict_proba(Xva_cat)[:, 1]
    thr, f1 = compute_best_threshold(yva, p_cat)

    cat_metrics.append({
        "fold": fold,
        "AUC": roc_auc_score(yva, p_cat),
        "best_threshold": thr,
        "fraud_F1": f1
    })
    cat_models.append(cat_model)

    # ===============================================================
    # 9B — LIGHTGBM
    # ===============================================================
    lgb_model = lgb.LGBMClassifier(
        n_estimators=700,
        learning_rate=0.05,
        max_depth=-1,
        num_leaves=64,
        subsample=0.9,
        colsample_bytree=0.9,
        random_state=SEED,
        verbose=-1     
    )
    lgb_model.fit(Xtr_lgb, ytr)

    p_lgb = lgb_model.predict_proba(Xva_lgb)[:, 1]
    thr, f1 = compute_best_threshold(yva, p_lgb)

    lgb_metrics.append({
        "fold": fold,
        "AUC": roc_auc_score(yva, p_lgb),
        "best_threshold": thr,
        "fraud_F1": f1
    })
    lgb_models.append(lgb_model)

    # ===============================================================
    # 9C — XGBOOST
    # ===============================================================
    xgb_model = xgb.XGBClassifier(
        n_estimators=700,
        learning_rate=0.05,
        max_depth=8,
        subsample=0.9,
        colsample_bytree=0.9,
        eval_metric="logloss",
        tree_method="hist",
        random_state=SEED,
        use_label_encoder=False
    )
    xgb_model.fit(Xtr_xgb, ytr)

    p_xgb = xgb_model.predict_proba(Xva_xgb)[:, 1]
    thr, f1 = compute_best_threshold(yva, p_xgb)

    xgb_metrics.append({
        "fold": fold,
        "AUC": roc_auc_score(yva, p_xgb),
        "best_threshold": thr,
        "fraud_F1": f1
    })
    xgb_models.append(xgb_model)

    # Memory clean
    gc.collect()


In [11]:
# ===============================================================
# SECTION 10 — COMPUTE ENSEMBLE WEIGHTS FROM CV F1 SCORES
# ===============================================================
# We use the out-of-fold F1 scores of each base model
# (CatBoost, LightGBM, XGBoost) to derive normalized ensemble
# weights. Models with higher F1 get slightly higher influence
# in the final probability blend.

# 1. Extract mean F1 for each model across all folds
cat_f1 = np.mean([m["fraud_F1"] for m in cat_metrics])
lgb_f1 = np.mean([m["fraud_F1"] for m in lgb_metrics])
xgb_f1 = np.mean([m["fraud_F1"] for m in xgb_metrics])

# 2. Sum of F1s (for normalization)
f1_sum = cat_f1 + lgb_f1 + xgb_f1

# 3. Normalized weights
w_cat = cat_f1 / f1_sum
w_lgb = lgb_f1 / f1_sum
w_xgb = xgb_f1 / f1_sum

print("Ensemble weights computed:")
print(f"  CatBoost:  {w_cat:.4f}")
print(f"  LightGBM: {w_lgb:.4f}")
print(f"  XGBoost:  {w_xgb:.4f}")


Ensemble weights computed:
  CatBoost:  0.3323
  LightGBM: 0.3341
  XGBoost:  0.3336


In [12]:
# ===============================================================
# SECTION 11 — CROSS-VALIDATED ENSEMBLE THRESHOLD TUNING
# ===============================================================

ensemble_thresholds = []
ensemble_f1s = []

gkf = GroupKFold(n_splits=5)

for fold, ((tr_idx, va_idx), cat_m, lgb_m, xgb_m) in enumerate(
    zip(
        gkf.split(X, y, groups),
        cat_models,
        lgb_models,
        xgb_models
    )
):

    print(f"\n========== ENSEMBLE FOLD {fold} ==========")

    # Extract fold splits
    X_train_fold = X.iloc[tr_idx].copy()
    y_train_fold = y.iloc[tr_idx].copy()

    X_val = X.iloc[va_idx].copy()
    y_val = y.iloc[va_idx].copy()

    # ----------------------------------------------------------
    # Apply fold-safe merchant encoding (same as Section 7)
    # ----------------------------------------------------------
    X_train_enc, X_val_enc = add_fold_safe_merchant_encodings(
        X_train_fold,
        X_val,
        y_train_fold
    )

    # Prepare model-specific views
    X_val_cat = X_val_enc.copy()
    X_val_lgb = X_val_enc.copy()
    X_val_xgb = X_val_enc.copy()

    # Apply label encoding for numeric models
    for col, le in label_encoders.items():
        X_val_lgb[col] = le.transform(X_val_lgb[col].astype(str))
        X_val_xgb[col] = le.transform(X_val_xgb[col].astype(str))

    # Predictions
    p_cat = cat_m.predict_proba(X_val_cat)[:, 1]
    p_lgb = lgb_m.predict_proba(X_val_lgb)[:, 1]
    p_xgb = xgb_m.predict_proba(X_val_xgb)[:, 1]

    # Weighted blend
    p_ens = w_cat * p_cat + w_lgb * p_lgb + w_xgb * p_xgb

    # Threshold optimization via PR-curve
    precision, recall, thresholds = precision_recall_curve(y_val, p_ens)
    f1_scores = 2 * precision * recall / (precision + recall + 1e-9)

    best_idx = np.argmax(f1_scores)
    best_thr = float(thresholds[best_idx])
    best_f1 = float(f1_scores[best_idx])

    ensemble_thresholds.append(best_thr)
    ensemble_f1s.append(best_f1)

# Final averaged results
ensemble_best_threshold = float(np.mean(ensemble_thresholds))
ensemble_best_f1 = float(np.mean(ensemble_f1s))

print("\n===== ENSEMBLE CV THRESHOLD =====")
print("Best Threshold:", ensemble_best_threshold)
print("Best Ensemble F1:", ensemble_best_f1)



========== ENSEMBLE FOLD 0 ==========

========== ENSEMBLE FOLD 1 ==========

========== ENSEMBLE FOLD 2 ==========

========== ENSEMBLE FOLD 3 ==========

========== ENSEMBLE FOLD 4 ==========

===== ENSEMBLE CV THRESHOLD =====
Best Threshold: 0.16603314614614537
Best Ensemble F1: 0.9376478623957523


In [13]:
# ===============================================================
# SECTION 12 — TRAIN FINAL MODELS ON FULL DATA
# ===============================================================

# 1. GENERATE MERCHANT FRAUD RATE FOR FULL TRAINING DATA
# We use the global logic (same as Section 13) to create the feature
temp_full = pd.DataFrame({"merchant_freq": X["merchant_freq"], "y": y})
train_bins = pd.qcut(temp_full["merchant_freq"], q=20, duplicates="drop")
fraud_rate_map = temp_full.groupby(train_bins)["y"].mean().to_dict()
global_mean_fraud = y.mean()

# Map the feature onto X
X_full = X.copy()
X_full["merchant_fraud_rate"] = X_full["merchant_freq"].apply(
    lambda x: next((rate for interval, rate in fraud_rate_map.items() if x in interval), global_mean_fraud)
)

# 2. Prepare CatBoost view
X_cat_full = X_full.copy()

# 3. Prepare LGB/XGB views with label-encoded categoricals
X_lgb_full = X_full.copy()
X_xgb_full = X_full.copy()

for col, le in label_encoders.items():
    # Ensure string type before encoding
    X_lgb_full[col] = le.transform(X_lgb_full[col].astype(str))
    X_xgb_full[col] = le.transform(X_xgb_full[col].astype(str))

# 4. Final CatBoost model
final_cat = CatBoostClassifier(
    iterations=700, depth=8, learning_rate=0.05, loss_function="Logloss",
    eval_metric="AUC", random_seed=SEED, verbose=False
)
final_cat.fit(X_cat_full, y, cat_features=cat_features, verbose=False)

# 5. Final LightGBM model
final_lgb = lgb.LGBMClassifier(
    n_estimators=700, learning_rate=0.05, max_depth=-1, num_leaves=64,
    subsample=0.9, colsample_bytree=0.9, random_state=SEED, verbose=-1
)
final_lgb.fit(X_lgb_full, y)

# 6. Final XGBoost model
final_xgb = xgb.XGBClassifier(
    n_estimators=700, learning_rate=0.05, max_depth=8, subsample=0.9,
    colsample_bytree=0.9, eval_metric="logloss", tree_method="hist",
    random_state=SEED, use_label_encoder=False
)
final_xgb.fit(X_xgb_full, y)

print("All final models successfully trained on full dataset.")

All final models successfully trained on full dataset.


In [14]:
# ===============================================================
# SECTION 13 — GLOBAL MERCHANT ENCODING ARTIFACTS
# ===============================================================
# For inference we need a stable, global mapping from merchant frequency bins to fraud rate.
# We construct this on the full training data and persist the
# artifacts to disk.

# 1. Build the full-data temp table for merchant frequency
temp_full = pd.DataFrame({
    "merchant_freq": X["merchant_freq"],
    "y": y
})

# 2. Global quantile bins over merchant frequency
train_bins = pd.qcut(
    temp_full["merchant_freq"],
    q=20,
    duplicates="drop"
)

# 3. Fraud rate per global frequency bin
fraud_rate_map = temp_full.groupby(train_bins)["y"].mean().to_dict()

# 4. Global mean fraud rate (fallback for unseen bins)
global_mean_fraud = float(y.mean())

print("Global merchant encoding artifacts built.")


Global merchant encoding artifacts built.


In [15]:
# ===============================================================
# SECTION 14 — EXPORT ALL ARTIFACTS FOR task1_predict.py
# ===============================================================
# We save:
#   - Final trained model objects (CatBoost, LightGBM, XGBoost).
#   - Merchant encoding artifacts (bins, fraud_rate_map, global mean).
#   - Ensemble weights and tuned threshold.
#   - Label encoders (for categorical columns).

# 1. Models
final_cat.save_model("final_cat.cbm")
final_lgb.booster_.save_model("final_lgb.txt")
final_xgb.save_model("final_xgb.json")

# 2. Merchant encoding artifacts
train_bins.to_pickle("train_bins.pkl")
pd.to_pickle(fraud_rate_map, "fraud_rate_map.pkl")
pd.to_pickle(global_mean_fraud, "global_mean_fraud.pkl")

# 3. Ensemble weights & best threshold
pd.to_pickle(w_cat, "w_cat.pkl")
pd.to_pickle(w_lgb, "w_lgb.pkl")
pd.to_pickle(w_xgb, "w_xgb.pkl")
pd.to_pickle(ensemble_best_threshold, "ensemble_best_threshold.pkl")

# 4. Label encoders for LGB/XGB categoricals
with open("label_encoders.pkl", "wb") as f:
    pickle.dump(label_encoders, f)

print("All artifacts exported successfully.")
for f in [
    "final_cat.cbm", "final_lgb.txt", "final_xgb.json",
    "train_bins.pkl", "fraud_rate_map.pkl", "global_mean_fraud.pkl",
    "w_cat.pkl", "w_lgb.pkl", "w_xgb.pkl",
    "ensemble_best_threshold.pkl", "label_encoders.pkl"
]:
    print(f, "→", os.path.exists(f))


All artifacts exported successfully.
final_cat.cbm → True
final_lgb.txt → True
final_xgb.json → True
train_bins.pkl → True
fraud_rate_map.pkl → True
global_mean_fraud.pkl → True
w_cat.pkl → True
w_lgb.pkl → True
w_xgb.pkl → True
ensemble_best_threshold.pkl → True
label_encoders.pkl → True
